In [372]:
from tqdm import tqdm
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterio.fill import fillnodata
from rasterio.windows import from_bounds, Window
from rasterio.transform import rowcol
from rasterio.plot import show
from rasterio.features import shapes

import xarray as xr
import rioxarray
from pysheds.grid import Grid

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

In [3]:
root = Path(os.getcwd())
data_root = root / "data"

In [173]:
dtm_dir = data_root / "DTM_rlp"
dtm_path = dtm_dir / "DTM Germany_Rheinland-Pfalz 20m.tif"
dtm_crop_path = dtm_dir / "dtm_crop.tif"

print(dtm_path)
print(dtm_crop_path)

c:\Users\Administrator\PythonProjects\abfluss_queich\data\DTM_rlp\DTM Germany_Rheinland-Pfalz 20m.tif
c:\Users\Administrator\PythonProjects\abfluss_queich\data\DTM_rlp\dtm_crop.tif


### Open hyras precip files and crop ROI

In [388]:
# Hyras path
hyras_dir = data_root / "precip_hyras"
hyras_paths = [p for p in hyras_dir.rglob("*.nc") if "v6-1" in p.stem] # subset for data at 6:00 morning

# Catchment path
basin_path = dtm_dir / "catchment_queich_siebeldingen.gpkg"

In [408]:
def read_rio_and_clip(
    ds: xr.core.dataset.Dataset, 
    basin: gpd.GeoDataFrame
    ):
    
    epsg = ds.crs.epsg_code
    precip = ds["pr"]
    
    # Reproject both to same crs
    precip = precip.rio.write_crs(epsg)
    basin = basin.to_crs(epsg)
    
    # Clip precip to basin contour
    precip_clip = precip.rio.clip(
        basin.geometry,
        drop=True
    )
    
    return precip_clip    

def aggregate_to_df(data):
    mean_data = data.mean(dim=["y", "x"], skipna=True)
    
    df_mean_data = mean_data.to_dataframe()
    df_mean_data = df_mean_data[["pr"]]
    
    return df_mean_data

In [414]:
# Basing
basin = gpd.read_file(basin_path)

dfs = []

# Iterate over hyras datasets
for path in tqdm(hyras_paths[1:]):
    ds = xr.open_dataset(path)
    
    precip_clip = read_rio_and_clip(ds=ds, basin=basin)
    df_mean_precip = aggregate_to_df(data=precip_clip)
    
    dfs.append(df_mean_precip)

# Concat dfs
df_concat = pd.concat(dfs, axis="index")

100%|██████████| 27/27 [03:59<00:00,  8.86s/it]


In [423]:
output_path = hyras_dir / "precip_mean_queich_watershed.csv"
df_concat.to_csv(path_or_buf=output_path, sep=",")